In [1]:
!pip install gradio requests pyperclip


In [2]:
import gradio as gr
import requests, re, pyperclip

# -------- GitHub Fetch --------
def fetch_from_github(query, max_files=1):
    try:
        r = requests.get(
            "https://api.github.com/search/code",
            headers={"Accept": "application/vnd.github+json"},
            params={"q": query, "per_page": max_files},
            timeout=10
        )
        data = r.json()
        samples = []
        for item in data.get("items", []):
            raw = item["html_url"].replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
            samples.append(requests.get(raw, timeout=10).text)
        return samples
    except:
        return []

# -------- Copy Helper --------
def copy_to_clipboard(text):
    pyperclip.copy(text)
    return "✅ Code copied to clipboard!"

# -------- Core Bug Fixer --------
def bug_fixer_ai(code_text, error_text, language):
    fixed_code = code_text
    explanation = "No fix applied."

    fetch_from_github(f"{language} {error_text} fix", 1)

    if language == "Python":
        if "ZeroDivisionError" in error_text:
            fixed_code = fixed_code.replace("/ 0", "/ len(nums)")
            explanation = "Division by zero fixed by using list length."
        elif "IndexError" in error_text:
            explanation = "Index out of range. Adjust loop boundary."
        elif "IndentationError" in error_text:
            fixed_code = fixed_code.replace("\nprint", "\n    print")
            explanation = "Indentation corrected."
        elif "expected ':'" in error_text or "invalid syntax" in error_text:
            fixed_code = re.sub(r'for (.*) in (.*)\n', r'for \1 in \2:\n', fixed_code)
            explanation = "Missing ':' in for-loop."
        elif "NameError" in error_text:
            explanation = "Variable not defined."

    elif language == "C":
        if "Segmentation fault" in error_text:
            explanation = "Invalid memory access. Pointer not initialized."
        elif "undeclared" in error_text:
            explanation = "Variable used without declaration."
        elif "semicolon" in error_text or ";" in error_text:
            fixed_code = fixed_code.replace(")", ");")
            explanation = "Missing semicolon."

    elif language == "C++":
        if "out_of_range" in error_text:
            explanation = "Vector index out of bounds."
        elif "not declared" in error_text:
            explanation = "Identifier not declared."
        elif "semicolon" in error_text:
            fixed_code = fixed_code.replace(")", ");")
            explanation = "Missing semicolon."

    elif language == "Java":
        if "NullPointerException" in error_text:
            explanation = "Object is null before usage."
        elif "ArrayIndexOutOfBoundsException" in error_text or "IndexOutOfBoundsException" in error_text:
            fixed_code = fixed_code.replace("i <= n", "i < n")
            explanation = "Loop boundary error fixed."
        elif "';' expected" in error_text:
            fixed_code = fixed_code.replace(")", ");")
            explanation = "Missing semicolon."
        elif "cannot find symbol" in error_text:
            explanation = "Method or variable not defined."

    return fixed_code, explanation

# -------- Examples --------
examples = {
    "Python": ("nums=[1,2,3]\nfor n in nums\nprint(n)", "SyntaxError: expected ':'"),
    "C": ("#include <stdio.h>\nint main(){printf(\"Hi\")\nreturn 0}", "expected ';'"),
    "C++": ("#include <iostream>\nusing namespace std;\nint main(){cout<<\"Hi\"\nreturn 0;}", "expected ';'"),
    "Java": ("public class T{public static void main(String[] a){System.out.println(\"Hi\")}}", "error: ';' expected")
}

def load_example(lang):
    return examples[lang][0], examples[lang][1]

def clear_all():
    return "", "", "", ""

# -------- UI Styling --------
custom_css = """
body {
    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
}
.gradio-container {
    color: #ffffff;
}
h2 {
    text-align: center;
    color: #00e5ff;
}
button {
    font-weight: bold;
    border-radius: 8px !important;
}

/* Generate Fix */
button:nth-of-type(1) { background:#00c853 !important; color:white !important; }

/* Load Example */
button:nth-of-type(2) { background:#2962ff !important; color:white !important; }

/* Clear All */
button:nth-of-type(3) { background:#d50000 !important; color:white !important; }

/* Copy Code */
button:nth-of-type(4) { background:#ff9100 !important; color:black !important; }
"""

# -------- UI --------
with gr.Blocks(theme=gr.themes.Glass(), css=custom_css) as ui:
    gr.Markdown("## 🤖 LLM–Based Bug Explanation & Code Fix Assistant")

    lang = gr.Dropdown(["Python", "C", "C++", "Java"], label="Select Language")
    code = gr.Textbox(lines=12, label="🧾 Buggy Code")
    error_in = gr.Textbox(lines=4, label="❗ Error Message")

    with gr.Row():
        run_btn = gr.Button("Generate Fix")
        example_btn = gr.Button("Load Example")
        clear_btn = gr.Button("Clear All")

    fixed_box = gr.Textbox(lines=14, label="Corrected Code")
    explain_box = gr.Textbox(lines=6, label="Explanation")

    copy_btn = gr.Button("Copy Code")
    status = gr.Textbox(label="Status")

    run_btn.click(bug_fixer_ai, [code, error_in, lang], [fixed_box, explain_box])
    example_btn.click(load_example, lang, [code, error_in])
    clear_btn.click(clear_all, None, [code, error_in, fixed_box, explain_box])
    copy_btn.click(copy_to_clipboard, fixed_box, status)

ui.launch()


/tmp/ipykernel_3012/3186432882.py:126: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Glass(), css=custom_css) as ui:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a8322708ad1a6a4df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
